[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S09/S09_01_clustering.ipynb)

# S09 · 01 — Segmentar huéspedes con clustering

**Tiempo estimado:** 1 h 30 min · **Unidad del temario:** 3.9 (clustering)

**Contexto TurisData.** TurisData Canarias tiene 1 200 huéspedes y hasta ahora
les manda a todos la misma newsletter. Dirección pregunta: *¿hay grupos de clientes
parecidos entre sí, y merece la pena tratarlos de forma distinta?*
Aquí **no hay etiquetas**: nadie nos ha dicho a qué grupo pertenece cada huésped.
Es aprendizaje **no supervisado**: dejamos que los datos se agrupen solos y luego
**nosotros** interpretamos qué significa cada grupo.

**Al terminar sabrás:**
1. Por qué hay que escalar las variables antes de agrupar.
2. Aplicar k-means y elegir el número de grupos con el coeficiente de silueta.
3. Leer un dendrograma y aplicar clustering jerárquico.
4. Usar DBSCAN y entender qué es el "ruido".
5. Poner nombre e interpretar cada segmento para el negocio.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
print(huespedes.shape)
huespedes.head()

Cada fila es un huésped con 7 variables de comportamiento (más su identificador).
Fíjate en las **escalas**: la edad va de 18 a 75, el gasto diario de 35 a 340 y
`viaja_con_ninos` solo vale 0 ó 1.

In [ ]:
huespedes.describe().round(1).T

## 1. Escalar: que todas las variables "pesen" lo mismo

**Idea.** Clustering agrupa por **distancia**. Si una variable tiene números
grandes (gasto en euros) y otra pequeños (0/1 de niños), la grande manda y la
pequeña no cuenta. Solución: **estandarizar** (restar la media y dividir por la
desviación típica) para que todas tengan media 0 y desviación 1.

**Ejemplo resuelto** con una sola columna:

In [ ]:
edad = huespedes["edad"]
edad_escalada = (edad - edad.mean()) / edad.std()
print("media:", round(edad_escalada.mean(), 3), "· desviación:", round(edad_escalada.std(), 3))

### Ejercicio 1
Crea la matriz `X` con **todas las variables menos `id_huesped`**, estandarizadas con
`StandardScaler` (usa `fit_transform`). `X` debe ser una matriz de 1200 filas y 7 columnas.
Guarda también los nombres de las variables en la lista `variables`.

In [ ]:
# TODO: Quita la columna id_huesped con drop(columns=...) y pásale el resultado a StandardScaler().fit_transform
...

assert X.shape == (1200, 7), "X debe tener 1200 filas y 7 columnas (sin id_huesped)"
assert abs(X.mean()) < 1e-6 and abs(X[:, 0].std() - 1) < 1e-6, "Revisa que estés escalando con StandardScaler"
print("Ejercicio 1 correcto ✔")

## 2. k-means: agrupar alrededor de centros

**Idea.** Le decimos cuántos grupos queremos (`k`). El algoritmo coloca `k` centros,
asigna cada huésped al centro más cercano, recalcula los centros y repite hasta que
no cambian. Es rápido y fácil de explicar. Su límite: **tú decides `k`** y prefiere
grupos "redondos" y de tamaño parecido.

**Ejemplo resuelto** con `k = 3`:

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=SEMILLA).fit(X)
print("Tamaño de cada grupo:", np.bincount(km3.labels_))
print("Inercia (suma de distancias al centro):", round(km3.inertia_))

### ¿Cuántos grupos? El coeficiente de silueta
Para cada huésped compara *lo cerca que está de su grupo* con *lo cerca que está del
grupo vecino*. Va de -1 a 1: **cuanto más alto, grupos más compactos y separados**.
Por encima de 0,5 se considera una buena estructura; entre 0,25 y 0,5, aceptable.

### Ejercicio 2
Calcula en el diccionario `siluetas` la silueta media (`silhouette_score(X, etiquetas)`)
de k-means para cada `k` de 2 a 8 (clave = `k`, valor = silueta). Después guarda en
`k_mejor` el `k` con silueta más alta. Usa `n_init=10` y `random_state=SEMILLA`.

In [ ]:
# TODO: Bucle for k in range(2, 9): entrena KMeans, saca .labels_ y calcula silhouette_score; luego max(siluetas, key=siluetas.get)
...

assert set(siluetas) == set(range(2, 9)), "siluetas debe tener las claves 2, 3, ..., 8"
assert 3 <= k_mejor <= 5, "Revisa: k_mejor debería ser el k con la silueta más alta (está entre 3 y 5)"
print("k con mejor silueta:", k_mejor)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(siluetas), list(siluetas.values()), marker="o")
ax.axvline(k_mejor, color="tab:red", linestyle="--", label=f"mejor k = {k_mejor}")
ax.set_xlabel("Número de grupos (k)")
ax.set_ylabel("Silueta media")
ax.set_title("¿Cuántos segmentos de huéspedes hay?")
ax.legend()
plt.show()

## 3. Interpretar: ¿quién es cada grupo?

**Idea.** Un grupo sin nombre no sirve al negocio. Calculamos la **media de cada
variable por grupo** y la leemos como si describiéramos a un cliente típico.

**Ejemplo resuelto:**

In [ ]:
km = KMeans(n_clusters=k_mejor, n_init=10, random_state=SEMILLA).fit(X)
huespedes["grupo"] = km.labels_
perfil = huespedes.groupby("grupo")[variables].mean().round(1)
perfil["n_huespedes"] = huespedes["grupo"].value_counts().sort_index()
perfil

In [ ]:
# Mapa de calor con los perfiles estandarizados (comparables entre variables)
perfil_z = pd.DataFrame(km.cluster_centers_, columns=variables)
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(perfil_z, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Perfil de cada grupo (en desviaciones típicas respecto a la media)")
ax.set_ylabel("Grupo")
plt.show()

### Ejercicio 3
Guarda en `grupo_familias` el número del grupo con **mayor proporción de huéspedes que
viajan con niños** (media de `viaja_con_ninos` por grupo) y en `grupo_aventura` el
número del grupo con **mayor `pct_gasto_actividades`** medio.

In [ ]:
# TODO: groupby("grupo")[columna].mean() y luego .idxmax()
...

assert huespedes[huespedes["grupo"] == grupo_familias]["viaja_con_ninos"].mean() > 0.6, "grupo_familias: busca el grupo donde casi todos viajan con niños"
assert huespedes[huespedes["grupo"] == grupo_aventura]["pct_gasto_actividades"].mean() > 50, "grupo_aventura: busca el grupo que más gasta en actividades"
print("Ejercicio 3 correcto ✔")

## 4. Clustering jerárquico: un árbol de parecidos

**Idea.** Empieza con cada huésped como su propio grupo y va **fusionando los dos más
parecidos** paso a paso. El resultado es un árbol (**dendrograma**): cuanto más alto es
el salto entre fusiones, más distintos eran los grupos que se unían. Se "corta" el árbol
a la altura que queramos. Con `linkage="ward"` se busca que los grupos sean compactos.

**Ejemplo resuelto:** dendrograma con una muestra de 150 huéspedes (con los 1 200 sería ilegible).

In [ ]:
muestra = np.random.default_rng(SEMILLA).choice(len(X), size=150, replace=False)
Z = linkage(X[muestra], method="ward")
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(Z, no_labels=True, color_threshold=Z[-4, 2], ax=ax)
ax.set_title("Dendrograma (150 huéspedes): los saltos grandes indican grupos distintos")
ax.set_ylabel("Distancia de fusión")
plt.show()

### Ejercicio 4
Aplica `AgglomerativeClustering` con `n_clusters=4` y `linkage="ward"` sobre **todos** los
huéspedes (`X`). Guarda las etiquetas en `etiquetas_jer` y su silueta en `silueta_jer`.

In [ ]:
# TODO: AgglomerativeClustering se configura con n_clusters y linkage; la silueta se calcula sobre X y las etiquetas
...

assert len(etiquetas_jer) == 1200 and len(set(etiquetas_jer)) == 4, "Debe haber 4 grupos y una etiqueta por huésped"
assert silueta_jer > 0.3, "La silueta debería ser superior a 0,3; revisa que uses X (escalado)"
print(f"Silueta jerárquico = {silueta_jer:.3f} · k-means con k={k_mejor}: {siluetas[k_mejor]:.3f}")

## 5. DBSCAN: grupos por densidad (y ruido)

**Idea.** No hay que decir cuántos grupos queremos. Define *"un grupo es una zona
densa de huéspedes"* con dos parámetros: `eps` (radio de vecindad) y `min_samples`
(cuántos vecinos hacen falta para considerarla densa). Lo que queda en zonas poco
densas se etiqueta **-1 = ruido** (huéspedes atípicos). Es útil para detectar
**casos raros**, pero es sensible a `eps`.

**Ejemplo resuelto:** probamos varios radios.

In [ ]:
for eps in (0.6, 0.8, 1.0, 1.2, 1.5):
    et = DBSCAN(eps=eps, min_samples=10).fit_predict(X)
    n_grupos = len(set(et)) - (1 if -1 in et else 0)
    print(f"eps={eps}: {n_grupos} grupos, {np.mean(et == -1):.0%} de ruido")

### Ejercicio 5
Entrena DBSCAN con `eps=1.0` y `min_samples=10`. Guarda las etiquetas en `etiquetas_db`,
el número de grupos encontrados (sin contar el ruido) en `n_grupos_db` y cuántos huéspedes
quedan como ruido en `n_ruido`.

In [ ]:
# TODO: El ruido tiene etiqueta -1: cuéntalo con (etiquetas == -1).sum(); los grupos son las etiquetas distintas menos el -1
...

assert n_grupos_db >= 3, "Con eps=1.0 debería encontrar varios grupos; revisa que uses X escalado"
assert 0 < n_ruido < 400, "n_ruido debe ser el número de huéspedes con etiqueta -1"
print(f"DBSCAN: {n_grupos_db} grupos y {n_ruido} huéspedes atípicos")

## Mini-reto integrador: ponle nombre a tus segmentos

1. Elige el método que te parezca mejor (k-means, jerárquico o DBSCAN) y justifícalo en una frase.
2. Para cada grupo, escribe **un nombre**, **una descripción de una línea** y **una acción comercial**.
3. Guarda los nombres en el diccionario `nombres_segmentos` (clave = número de grupo de k-means).

*Ejemplo:* `{0: "Familias de verano"}`. Este trabajo alimenta el reto "Conocer al huésped".

In [ ]:
nombres_segmentos = {}   # rellena: {0: "...", 1: "...", ...}
# TODO: Mira el mapa de calor: edad, gasto, niños y % de gasto en actividades dan mucha pista
...

assert len(nombres_segmentos) == k_mejor, "Pon un nombre a cada uno de los grupos de k-means"
print(nombres_segmentos)

## Preguntas de reflexión
1. ¿Qué pasaría con los resultados si no escalaras las variables? Pruébalo y compara la silueta.
2. ¿Por qué la silueta más alta no siempre da el mejor número de segmentos *para el negocio*?
3. ¿Qué método te parece más fácil de explicar a dirección? ¿Y el más útil para detectar clientes raros?
4. Un segmento "familias" sale de datos de comportamiento, no de una etiqueta oficial: ¿qué riesgos hay
   si el hotel usa el segmento para decidir a quién ofrece descuentos? *(pista: sesgos y discriminación)*.

## Qué has aprendido
- Escalar antes de medir distancias.
- **k-means** (rápido, exige `k`), **jerárquico** (dendrograma, más lento) y **DBSCAN** (densidad, detecta ruido).
- La **silueta** ayuda a elegir `k`, pero la interpretación la pone el negocio.
- Un segmento solo es útil si tiene nombre, tamaño y una acción asociada.